# 🛡️ Phishing URL Detection — Complete ML Pipeline
### University Project

**Strategy:** Feature *Selection* (not engineering) — the dataset is already feature-engineered by design.  
We select the most discriminative subset to avoid the **Curse of Dimensionality**.

| Section | Topic |
|---------|-------|
| 1 | Imports & Setup |
| 2 | Data Loading & Profiling |
| 3 | Exploratory Data Analysis |
| 4 | Preprocessing |
| 5 | Class Imbalance |
| 6 | Curse of Dimensionality Analysis |
| 7 | Feature Selection (the reduction step) |
| 8 | Model Training + Cross-Validation |
| 9 | Model Evaluation |
| 10 | Final Summary |

---
## 1. 📦 Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# Preprocessing & Pipeline
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, learning_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Dimensionality & Feature Selection
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import (SelectKBest, f_classif,
                                        mutual_info_classif, RFECV)
from sklearn.metrics.pairwise import euclidean_distances

# Models
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import (RandomForestClassifier,
                                     AdaBoostClassifier,
                                     GradientBoostingClassifier)

# Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              ConfusionMatrixDisplay)

# Optional: XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print('✅ XGBoost found')
except ImportError:
    XGB_AVAILABLE = False
    print('⚠️  XGBoost not found → pip install xgboost   (using GradientBoosting instead)')

# Optional: SMOTE
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print('✅ imbalanced-learn found')
except ImportError:
    SMOTE_AVAILABLE = False
    print('⚠️  imbalanced-learn not found → pip install imbalanced-learn   (using class_weight instead)')

# Style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold'})
C = {'phish': '#e74c3c', 'legit': '#2ecc71', 'blue': '#3498db', 'orange': '#f39c12'}

RANDOM_STATE = 42
print('\n✅ All imports successful')

---
## 2. 📂 Data Loading & Profiling

In [ ]:
# ─── Load Data ───────────────────────────────────────────────────────────────
# 👉 UPDATE this path to your CSV file
df_raw = pd.read_csv('your_dataset.csv')   # <── CHANGE THIS

# Drop non-feature columns
drop_cols = ['URL', 'Title', 'Domain']     # identifiers / text cols
df_raw = df_raw.drop(columns=[c for c in drop_cols if c in df_raw.columns])

print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# ─── 2.1  Basic Profile ───────────────────────────────────────────────────────
print('═'*55)
print('  DATASET PROFILE')
print('═'*55)
print(f'  Rows             : {df_raw.shape[0]:,}')
print(f'  Columns          : {df_raw.shape[1]}')
print(f'  Memory           : {df_raw.memory_usage(deep=True).sum()/1024:.1f} KB')
print(f'  Duplicate rows   : {df_raw.duplicated().sum()}')
print(f'  Missing values   : {df_raw.isnull().sum().sum()}')
print()

# dtype breakdown
print('  Data types:')
print(df_raw.dtypes.value_counts().to_string())

# Binary vs continuous
feat_cols   = [c for c in df_raw.columns if c != 'label']
binary_cols = [c for c in feat_cols if df_raw[c].nunique() <= 2]
cont_cols   = [c for c in feat_cols if c not in binary_cols]
print(f'\n  Binary features     : {len(binary_cols)}')
print(f'  Continuous features : {len(cont_cols)}')
print(f'  Total features      : {len(feat_cols)}')

In [ ]:
# ─── 2.2  Descriptive Statistics ──────────────────────────────────────────────
print('Descriptive Statistics:')
display(
    df_raw.describe().T
    .style.background_gradient(cmap='Blues', subset=['mean','std'])
    .format('{:.3f}')
)

In [ ]:
# ─── 2.3  Missing Values ──────────────────────────────────────────────────────
missing = df_raw.isnull().sum()

fig, ax = plt.subplots(figsize=(12, 3))
missing_pct = (missing / len(df_raw)) * 100
if missing_pct.max() == 0:
    ax.text(0.5, 0.5, '✅  No missing values in the dataset!',
            ha='center', va='center', fontsize=16, color='green',
            transform=ax.transAxes)
else:
    missing_pct[missing_pct > 0].plot(kind='bar', ax=ax, color=C['phish'])
    ax.set_ylabel('% Missing')
ax.set_title('Missing Value Check')
plt.tight_layout(); plt.show()

In [ ]:
# ─── 2.4  Skewness & Kurtosis ─────────────────────────────────────────────────
sk_df = pd.DataFrame({
    'Skewness' : df_raw[feat_cols].skew(),
    'Kurtosis' : df_raw[feat_cols].kurtosis()
}).sort_values('Skewness', key=abs, ascending=False)

print('Top 15 Most Skewed Features:')
display(sk_df.head(15).style
        .background_gradient(cmap='RdYlGn_r', subset=['Skewness'])
        .format('{:.3f}'))

fig, ax = plt.subplots(figsize=(14, 4))
clrs = ['#e74c3c' if abs(v)>1 else '#f39c12' if abs(v)>0.5 else '#2ecc71'
        for v in sk_df['Skewness']]
ax.bar(range(len(sk_df)), sk_df['Skewness'], color=clrs, edgecolor='black', lw=0.3)
ax.set_xticks(range(len(sk_df)))
ax.set_xticklabels(sk_df.index, rotation=90, fontsize=7)
ax.axhline(0, color='black', lw=1)
for h, ls, lbl in [(1,'--','High |skew|>1'), (0.5,':','Moderate |skew|>0.5')]:
    ax.axhline(h, color='red' if h==1 else 'orange', ls=ls, lw=1.2, label=lbl)
    ax.axhline(-h, color='red' if h==1 else 'orange', ls=ls, lw=1.2)
ax.set_title('Skewness per Feature  (🔴 high  🟠 moderate  🟢 low)')
ax.set_ylabel('Skewness')
ax.legend()
plt.tight_layout(); plt.show()

---
## 3. 📊 Exploratory Data Analysis (EDA)

In [ ]:
# ─── 3.1  Class Distribution ──────────────────────────────────────────────────
label_map   = {0: 'Phishing', 1: 'Legitimate'}
cnt         = df_raw['label'].value_counts().sort_index()
imb_ratio   = cnt.max() / cnt.min()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Class Distribution', fontsize=14)

axes[0].bar([label_map[k] for k in cnt.index], cnt.values,
             color=[C['phish'], C['legit']], edgecolor='black', width=0.45)
for i, v in enumerate(cnt.values):
    axes[0].text(i, v+5, str(v), ha='center', fontweight='bold')
axes[0].set_ylabel('Count'); axes[0].set_title('Sample Counts')

axes[1].pie(cnt.values, labels=[label_map[k] for k in cnt.index],
             colors=[C['phish'], C['legit']], autopct='%1.1f%%',
             startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Proportion')

plt.tight_layout(); plt.show()
print(f'Phishing   : {cnt.get(0,0):,}')
print(f'Legitimate : {cnt.get(1,0):,}')
print(f'Imbalance  : {imb_ratio:.2f} : 1')

In [ ]:
# ─── 3.2  Continuous Feature Distributions ────────────────────────────────────
plot_cont = cont_cols[:12]   # first 12 for readability
rows, cols_n = (len(plot_cont)+3)//4, 4

fig, axes = plt.subplots(rows, cols_n, figsize=(18, rows*3.5))
fig.suptitle('Continuous Feature Distributions: Phishing vs Legitimate', fontsize=14)
axes = axes.flatten()

for i, feat in enumerate(plot_cont):
    for lbl, col, nm in [(0,C['phish'],'Phishing'),(1,C['legit'],'Legitimate')]:
        d = df_raw[df_raw['label']==lbl][feat].dropna()
        axes[i].hist(d, bins=30, alpha=0.55, color=col, label=nm, edgecolor='none')
        axes[i].axvline(d.mean(), color=col, ls='--', lw=1.5)
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ─── 3.3  Box Plots ───────────────────────────────────────────────────────────
plot_box = cont_cols[:8]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Box Plots by Class', fontsize=14)
axes = axes.flatten()

for i, feat in enumerate(plot_box):
    bp = axes[i].boxplot(
        [df_raw[df_raw['label']==0][feat].dropna(),
         df_raw[df_raw['label']==1][feat].dropna()],
        patch_artist=True, labels=['Phishing','Legitimate'],
        medianprops=dict(color='black', lw=2))
    bp['boxes'][0].set_facecolor(C['phish']+'70')
    bp['boxes'][1].set_facecolor(C['legit']+'70')
    axes[i].set_title(feat, fontsize=9)

plt.tight_layout(); plt.show()

In [ ]:
# ─── 3.4  Binary Feature Rates ────────────────────────────────────────────────
rows_b, cols_b = (len(binary_cols)+3)//4, 4

fig, axes = plt.subplots(rows_b, cols_b, figsize=(18, rows_b*3.5))
fig.suptitle('Binary Feature: % = 1 per Class', fontsize=14)
axes = axes.flatten()

for i, feat in enumerate(binary_cols):
    p_rate = df_raw[df_raw['label']==0][feat].mean()*100
    l_rate = df_raw[df_raw['label']==1][feat].mean()*100
    bars = axes[i].bar(['Phishing','Legitimate'], [p_rate, l_rate],
                        color=[C['phish'], C['legit']], edgecolor='black')
    axes[i].set_title(feat, fontsize=8)
    axes[i].set_ylim(0, 120)
    for b in bars:
        axes[i].text(b.get_x()+b.get_width()/2, b.get_height()+2,
                     f'{b.get_height():.0f}%', ha='center', fontsize=8)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ─── 3.5  Correlation Heatmap ─────────────────────────────────────────────────
corr = df_raw[feat_cols + ['label']].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.3,
            annot_kws={'size': 6}, cbar_kws={'shrink': 0.7})
ax.set_title('Full Feature Correlation Heatmap', fontsize=15, pad=15)
plt.tight_layout(); plt.show()

In [ ]:
# ─── 3.6  Feature-Label Correlation Bar ───────────────────────────────────────
corr_label = df_raw[feat_cols + ['label']].corr()['label'].drop('label').sort_values()

fig, ax = plt.subplots(figsize=(10, 14))
clrs = [C['phish'] if v < 0 else C['legit'] for v in corr_label]
ax.barh(corr_label.index, corr_label.values, color=clrs, edgecolor='black', lw=0.3)
ax.axvline(0, color='black', lw=1.5)
ax.set_title('Pearson Correlation with Label\n(Red = phishing indicator | Green = legit indicator)', fontsize=13)
ax.set_xlabel('Correlation')
plt.tight_layout(); plt.show()

print('Top 5 Phishing Indicators:')
print(corr_label.head(5).round(3).to_string())
print('\nTop 5 Legit Indicators:')
print(corr_label.tail(5).round(3).to_string())

In [ ]:
# ─── 3.7  Statistical Significance (Welch T-Test) ────────────────────────────
print(f'{"Feature":<38} {"t-stat":>9} {"p-value":>11} {"Sig?":>8}')
print('─'*70)

sig_feats = []
for feat in feat_cols:
    g0 = df_raw[df_raw['label']==0][feat].dropna()
    g1 = df_raw[df_raw['label']==1][feat].dropna()
    t, p = stats.ttest_ind(g0, g1, equal_var=False)
    sig  = '✅ YES' if p < 0.05 else '❌ NO'
    if p < 0.05: sig_feats.append(feat)
    print(f'{feat:<38} {t:>9.3f} {p:>11.5f} {sig:>8}')

print(f'\n✅ {len(sig_feats)}/{len(feat_cols)} features show significant class difference (p<0.05)')

---
## 4. 🧹 Preprocessing

In [ ]:
# ─── 4.1  Separate X / y ──────────────────────────────────────────────────────
df = df_raw.copy()

X_all = df.drop(columns=['label'])
y_all = df['label']

print(f'X shape : {X_all.shape}')
print(f'y shape : {y_all.shape}')
print(f'Features: {list(X_all.columns)}')

In [ ]:
# ─── 4.2  Handle Missing Values ───────────────────────────────────────────────
print(f'Missing before: {X_all.isnull().sum().sum()}')
X_all = X_all.fillna(X_all.median(numeric_only=True))
print(f'Missing after : {X_all.isnull().sum().sum()}')

In [ ]:
# ─── 4.3  Remove Duplicates ───────────────────────────────────────────────────
df_clean       = pd.concat([X_all, y_all], axis=1).drop_duplicates()
X_all          = df_clean.drop(columns=['label'])
y_all          = df_clean['label']
print(f'Rows after dedup: {len(X_all)}')

In [ ]:
# ─── 4.4  Outlier Capping (IQR Winsorization — continuous only) ──────────────
X_clean = X_all.copy()
total_capped = 0

for col in cont_cols:
    if col not in X_clean.columns: continue
    Q1, Q3 = X_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_cap = ((X_clean[col] < lo) | (X_clean[col] > hi)).sum()
    total_capped += n_cap
    X_clean[col] = X_clean[col].clip(lo, hi)

X_all = X_clean.copy()
print(f'Total outlier values capped (Winsorization): {total_capped}')
print('✅ Binary features left untouched')

In [ ]:
# ─── 4.5  Train / Test Split (stratified 80/20) ───────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all
)

print(f'Train : {X_train.shape}  |  {dict(y_train.value_counts().sort_index())}')
print(f'Test  : {X_test.shape}   |  {dict(y_test.value_counts().sort_index())}')

# Scale (needed for LR and PCA)
scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)   # fit on train only!
X_test_sc   = scaler.transform(X_test)         # transform test
print('✅ StandardScaler fitted on training set only (no data leakage)')

---
## 5. ⚖️ Class Imbalance

In [ ]:
# ─── 5.1  Measure Imbalance ───────────────────────────────────────────────────
counts    = y_train.value_counts().sort_index()
imb_ratio = counts.max() / counts.min()

print('═'*45)
print('  CLASS IMBALANCE (Training Set)')
print('═'*45)
print(f'  Phishing   (0) : {counts.get(0,0):,}')
print(f'  Legitimate (1) : {counts.get(1,0):,}')
print(f'  Ratio          : {imb_ratio:.2f} : 1')

if imb_ratio < 1.5:
    print('\n  ✅ Balanced — using class_weight="balanced" as best-practice')
elif imb_ratio < 4:
    print('\n  ⚠️  Moderate imbalance — SMOTE recommended')
else:
    print('\n  ❌ Severe imbalance — SMOTE strongly recommended')

cw_arr  = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = dict(enumerate(cw_arr))
print(f'  Class weights  : {cw_dict}')

In [ ]:
# ─── 5.2  Apply SMOTE (if available) ─────────────────────────────────────────
if SMOTE_AVAILABLE and imb_ratio >= 1.5:
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    X_train_res_sc = scaler.transform(X_train_res)   # scale resampled

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, (title, ydata) in zip(axes, [
        ('Before SMOTE', y_train), ('After SMOTE', y_train_res)
    ]):
        c = pd.Series(ydata).value_counts().sort_index()
        ax.bar([label_map[k] for k in c.index], c.values,
               color=[C['phish'],C['legit']], edgecolor='black')
        ax.set_title(title); ax.set_ylabel('Count')
        for i,v in enumerate(c.values):
            ax.text(i, v+5, str(v), ha='center', fontweight='bold')
    plt.tight_layout(); plt.show()

    USE_SMOTE = True
    print('✅ SMOTE applied')
else:
    X_train_res, y_train_res    = X_train.copy(), y_train.copy()
    X_train_res_sc              = X_train_sc.copy()
    USE_SMOTE                   = False
    print('Using class_weight="balanced" — SMOTE skipped')

---
## 6. 🌌 Curse of Dimensionality Analysis

> **Why this matters:** With too many features and too few samples, distances between points become meaningless, models overfit, and generalisation collapses. We must quantify this before selecting features.

In [ ]:
# ─── 6.1  Samples-to-Features Ratio ──────────────────────────────────────────
n_samples  = X_train.shape[0]
n_features = X_train.shape[1]
ratio      = n_samples / n_features

print('═'*52)
print('  CURSE OF DIMENSIONALITY — KEY NUMBERS')
print('═'*52)
print(f'  Training samples : {n_samples:,}')
print(f'  Features         : {n_features}')
print(f'  Sample:Feature   : {ratio:.1f} : 1')
print()
print('  Rule-of-thumb thresholds:')
print(f'  • Good (>30:1)     → need {n_features*30:,} samples   ', '✅' if ratio >= 30 else '❌')
print(f'  • Acceptable(>10:1)→ need {n_features*10:,} samples  ', '✅' if ratio >= 10 else '❌')
print(f'  • Risky (<10:1)    → current ratio = {ratio:.1f}       ', '⚠️'  if ratio < 10 else '✅')
print()

# How many features should we ideally have?
ideal_max = n_samples // 30
acceptable_max = n_samples // 10
print(f'  → For {n_samples} samples, ideal max features   : {ideal_max}')
print(f'  → For {n_samples} samples, acceptable max feats : {acceptable_max}')
print(f'  → We currently have                           : {n_features} features')
print(f'  → We should reduce to around                  : {acceptable_max}–{ideal_max} features')

In [ ]:
# ─── 6.2  Distance Concentration (empirical proof) ───────────────────────────
# As dimensions grow, all pairwise distances converge → nearest neighbour becomes meaningless

np.random.seed(RANDOM_STATE)
sample_idx  = np.random.choice(len(X_train_sc), min(300, len(X_train_sc)), replace=False)
X_sample    = X_train_sc[sample_idx]

max_d = n_features
dim_range   = list(range(1, max_d+1, max(1, max_d//20)))   # ~20 checkpoints
if max_d not in dim_range: dim_range.append(max_d)

contrasts, means, stds = [], [], []
for d in dim_range:
    dists  = euclidean_distances(X_sample[:, :d]).flatten()
    dists  = dists[dists > 0]
    dmax, dmin, dmean = dists.max(), dists.min(), dists.mean()
    contrasts.append((dmax - dmin) / (dmean + 1e-9))
    means.append(dmean)
    stds.append(dists.std())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curse of Dimensionality — Empirical Evidence', fontsize=14)

axes[0].plot(dim_range, contrasts, 'o-', color=C['phish'], lw=2.5, ms=5)
axes[0].fill_between(dim_range, contrasts, alpha=0.12, color=C['phish'])
axes[0].set_xlabel('Number of Dimensions (Features)')
axes[0].set_ylabel('Relative Distance Contrast  (max−min)/mean')
axes[0].set_title('Distance Contrast Drops as Dimensions Grow\n'
                   '→ All points become equidistant (bad for KNN, SVM, etc.)')
# Mark current full-dim value
axes[0].axvline(max_d, color='gray', ls='--', lw=1.2,
                label=f'Current ({max_d} features)')
axes[0].legend()

axes[1].plot(dim_range, means, 'o-', color=C['blue'], lw=2.5, ms=5, label='Mean dist')
axes[1].fill_between(dim_range,
                      np.array(means)-np.array(stds),
                      np.array(means)+np.array(stds),
                      alpha=0.15, color=C['blue'])
axes[1].set_xlabel('Number of Dimensions')
axes[1].set_ylabel('Average Pairwise Distance')
axes[1].set_title('Mean Distance Grows with Dimensions\n→ Data becomes sparse')
axes[1].legend()

plt.tight_layout(); plt.show()
print('💡 Takeaway: reducing features IMPROVES distance-based learning & reduces noise for all models.')

In [ ]:
# ─── 6.3  PCA Variance — How many components capture useful info? ─────────────
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_train_sc)

cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
n90    = int(np.argmax(cumvar >= 90) + 1)
n95    = int(np.argmax(cumvar >= 95) + 1)
n99    = int(np.argmax(cumvar >= 99) + 1)

print(f'PCA components to explain 90% variance : {n90}  (out of {n_features})')
print(f'PCA components to explain 95% variance : {n95}  (out of {n_features})')
print(f'PCA components to explain 99% variance : {n99}  (out of {n_features})')
print(f'\n→ ~{n95} meaningful dimensions in the data (rest is noise/redundancy)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PCA Variance Analysis — Effective Dimensionality of the Data', fontsize=13)

# Scree plot
axes[0].bar(range(1, min(31, n_features+1)),
             pca_full.explained_variance_ratio_[:30]*100,
             color=C['blue'], edgecolor='black', lw=0.3)
axes[0].set_title('Individual Variance per PC (Scree Plot)')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')

# Cumulative
axes[1].plot(range(1, len(cumvar)+1), cumvar, lw=2.5, color=C['blue'])
axes[1].fill_between(range(1, len(cumvar)+1), cumvar, alpha=0.12, color=C['blue'])
for th, col, n in [(90,C['legit'],n90),(95,C['orange'],n95),(99,C['phish'],n99)]:
    axes[1].axhline(th, color=col, ls='--', lw=1.5, label=f'{th}% → {n} PCs')
    axes[1].axvline(n,  color=col, ls=':',  lw=1)
axes[1].set_title('Cumulative Explained Variance (Elbow Method)')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# ─── 6.4  t-SNE — Visual proof that classes ARE separable in low dimensions ───
print('Running t-SNE... (may take ~1 min)')
n_tsne = min(1000, len(X_train_sc))
idx    = np.random.choice(len(X_train_sc), n_tsne, replace=False)

# Pre-reduce with PCA (speeds up t-SNE)
pca_pre = PCA(n_components=min(30, n_features), random_state=RANDOM_STATE)
X_pre   = pca_pre.fit_transform(X_train_sc[idx])

tsne    = TSNE(n_components=2, perplexity=30, n_iter=1000,
               random_state=RANDOM_STATE, verbose=0)
X_2d    = tsne.fit_transform(X_pre)
y_2d    = y_train.values[idx]

fig, ax = plt.subplots(figsize=(9, 7))
for lbl, col, nm in [(0,C['phish'],'Phishing'),(1,C['legit'],'Legitimate')]:
    m = y_2d == lbl
    ax.scatter(X_2d[m,0], X_2d[m,1], c=col, alpha=0.45, s=15,
               label=nm, edgecolors='none')
ax.set_title('t-SNE 2D Projection — Classes are separable even in 2D!\n'
              '(Confirms that a small feature subset will work well)', fontsize=12)
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.legend(fontsize=11, markerscale=2)
plt.tight_layout(); plt.show()

---
## 7. ✂️ Feature Selection

We use **three independent methods** and take the **union of top features** agreed upon by at least 2 methods.  
Target: reduce to **`K_FINAL`** features — determined by the samples:features analysis above.

In [ ]:
# ─── 7.0  Decide target K (from CoD analysis) ─────────────────────────────────
# acceptable_max was computed in Section 6.1
# Cap at 20 max for good model performance
K_FINAL = min(acceptable_max, 20)
K_FINAL = max(K_FINAL, 10)    # at least 10 features

print(f'Target feature count K = {K_FINAL}')
print(f'(Justified by samples:features ratio — ensures we stay out of CoD zone)')

In [ ]:
# ─── 7.1  Method 1: ANOVA F-Score ─────────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2

f_sel   = SelectKBest(f_classif, k='all')
f_sel.fit(X_train_res, y_train_res)
f_scores = pd.Series(f_sel.scores_, index=X_train.columns).sort_values(ascending=False)

print('Top features by ANOVA F-Score:')
print(f_scores.head(K_FINAL).round(2).to_string())

In [ ]:
# ─── 7.2  Method 2: Mutual Information ────────────────────────────────────────
mi_scores = mutual_info_classif(X_train_res, y_train_res, random_state=RANDOM_STATE)
mi_scores = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

print('Top features by Mutual Information:')
print(mi_scores.head(K_FINAL).round(4).to_string())

In [ ]:
# ─── 7.3  Method 3: Random Forest Feature Importance ─────────────────────────
rf_selector = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_selector.fit(X_train_res, y_train_res)
rf_scores = pd.Series(rf_selector.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print('Top features by RF Importance:')
print(rf_scores.head(K_FINAL).round(5).to_string())

In [ ]:
# ─── 7.4  Visualise All Three Methods ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 9))
fig.suptitle(f'Feature Selection — Top {K_FINAL} Features by 3 Methods', fontsize=14)

for ax, (scores, title) in zip(axes, [
    (f_scores.head(K_FINAL),  'ANOVA F-Score'),
    (mi_scores.head(K_FINAL), 'Mutual Information'),
    (rf_scores.head(K_FINAL), 'RF Importance')
]):
    clrs = [C['phish'] if i < 5 else C['blue'] for i in range(len(scores))]
    scores.sort_values().plot(kind='barh', ax=ax, color=clrs[::-1],
                               edgecolor='black', lw=0.3)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Score')

plt.tight_layout(); plt.show()

In [ ]:
# ─── 7.5  Select Final Features (voting across 3 methods) ────────────────────
top_f  = set(f_scores.head(K_FINAL).index)
top_mi = set(mi_scores.head(K_FINAL).index)
top_rf = set(rf_scores.head(K_FINAL).index)

# Features that appear in at least 2 out of 3 methods
from collections import Counter
all_top = list(top_f) + list(top_mi) + list(top_rf)
vote_count = Counter(all_top)
selected_features = [f for f, v in vote_count.items() if v >= 2]

# If we still have too many, take top K_FINAL by RF as tie-breaker
if len(selected_features) > K_FINAL:
    selected_features = [f for f in rf_scores.index if f in selected_features][:K_FINAL]

# If too few, pad with highest RF scores
if len(selected_features) < 8:
    selected_features = list(rf_scores.head(K_FINAL).index)

print(f'✅ Final selected features ({len(selected_features)} total):')
for f in selected_features:
    votes = vote_count[f]
    print(f'   [{votes}/3 methods] {f}')

In [ ]:
# ─── 7.6  Voting Agreement Heatmap ────────────────────────────────────────────
# Show rank of each feature in each method
rank_df = pd.DataFrame({
    'F-Score Rank' : f_scores.rank(ascending=False),
    'MI Rank'      : mi_scores.rank(ascending=False),
    'RF Rank'      : rf_scores.rank(ascending=False),
}).loc[selected_features].sort_values('RF Rank')

fig, ax = plt.subplots(figsize=(9, max(5, len(selected_features)*0.35)))
sns.heatmap(rank_df, annot=True, fmt='.0f', cmap='YlOrRd_r',
             ax=ax, linewidths=0.4, cbar_kws={'label': 'Rank (1=best)'})
ax.set_title('Feature Rank Across 3 Selection Methods\n(Lower rank = more important)',
              fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ─── 7.7  Apply Selection — Build Final Datasets ──────────────────────────────
# These are the ONLY datasets used for model training from here on
X_train_sel    = X_train_res[selected_features].values     # tree-based models
X_test_sel     = X_test[selected_features].values
y_train_sel    = y_train_res.values
y_test_arr     = y_test.values

# Scaled version for Logistic Regression
sc_final       = StandardScaler()
X_train_sel_sc = sc_final.fit_transform(X_train_sel)
X_test_sel_sc  = sc_final.transform(X_test_sel)

new_ratio = len(X_train_sel) / len(selected_features)
print(f'Before selection : {n_features} features  |  sample:feature = {n_samples/n_features:.1f}:1')
print(f'After  selection : {len(selected_features)} features  |  sample:feature = {new_ratio:.1f}:1')
print(f'\n✅ Now well outside the Curse of Dimensionality danger zone!')

---
## 8. 🤖 Model Training & Cross-Validation

In [ ]:
# ─── 8.1  Define Models ───────────────────────────────────────────────────────
if XGB_AVAILABLE:
    boost_model = xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    )
    boost_name = 'XGBoost'
else:
    boost_model = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        subsample=0.8, random_state=RANDOM_STATE
    )
    boost_name = 'GradientBoosting'

# Each model gets the right X (scaled for LR, raw for trees)
models = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, class_weight='balanced',
                                     C=1.0, random_state=RANDOM_STATE),
        'Xtr': X_train_sel_sc, 'Xte': X_test_sel_sc
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(max_depth=8, class_weight='balanced',
                                         min_samples_leaf=5, random_state=RANDOM_STATE),
        'Xtr': X_train_sel, 'Xte': X_test_sel
    },
    'Random Forest': {
        'model': RandomForestClassifier(n_estimators=200, max_depth=12,
                                         class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
        'Xtr': X_train_sel, 'Xte': X_test_sel
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(n_estimators=150, learning_rate=0.5,
                                     random_state=RANDOM_STATE),
        'Xtr': X_train_sel, 'Xte': X_test_sel
    },
    boost_name: {
        'model': boost_model,
        'Xtr': X_train_sel, 'Xte': X_test_sel
    }
}

print(f'Models: {list(models.keys())}')
print(f'Training on {len(selected_features)} selected features')

In [ ]:
# ─── 8.2  5-Fold Stratified Cross-Validation ──────────────────────────────────
cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_metrics  = ['accuracy','precision','recall','f1','roc_auc']

cv_results  = {}
trained     = {}

print('Running 5-Fold Stratified Cross-Validation...')
print('═'*65)

for name, info in models.items():
    print(f'\n  ▶ {name}')
    scores = cross_validate(
        info['model'], info['Xtr'], y_train_sel,
        cv=cv, scoring=cv_metrics,
        return_train_score=True, n_jobs=-1
    )
    info['model'].fit(info['Xtr'], y_train_sel)   # final fit on full train
    trained[name] = info['model']

    cv_results[name] = {
        'CV Accuracy' : scores['test_accuracy'].mean(),
        'CV Precision': scores['test_precision'].mean(),
        'CV Recall'   : scores['test_recall'].mean(),
        'CV F1'       : scores['test_f1'].mean(),
        'CV AUC'      : scores['test_roc_auc'].mean(),
        'CV Acc Std'  : scores['test_accuracy'].std(),
        'Train Acc'   : scores['train_accuracy'].mean(),
        'Overfit Gap' : scores['train_accuracy'].mean() - scores['test_accuracy'].mean()
    }
    r = cv_results[name]
    print(f'    Accuracy  : {r["CV Accuracy"]:.4f} ± {r["CV Acc Std"]:.4f}')
    print(f'    F1-Score  : {r["CV F1"]:.4f}')
    print(f'    AUC-ROC   : {r["CV AUC"]:.4f}')
    print(f'    Overfit   : {r["Overfit Gap"]:.4f}  {"⚠️" if r["Overfit Gap"]>0.05 else "✅"}')

cv_df = pd.DataFrame(cv_results).T
print('\n── Cross-Validation Summary ──')
display(
    cv_df[['CV Accuracy','CV Precision','CV Recall','CV F1','CV AUC','Overfit Gap']]
    .round(4)
    .style
    .highlight_max(color='#c6efce', subset=['CV Accuracy','CV F1','CV AUC'])
    .highlight_min(color='#ffc7ce', subset=['Overfit Gap'])
    .format('{:.4f}')
)

---
## 9. 📈 Model Evaluation

In [ ]:
# ─── 9.1  Test Set Predictions ────────────────────────────────────────────────
test_results = {}
preds        = {}

for name, info in models.items():
    y_pred = trained[name].predict(info['Xte'])
    y_prob = trained[name].predict_proba(info['Xte'])[:, 1]
    preds[name] = {'y_pred': y_pred, 'y_prob': y_prob}
    test_results[name] = {
        'Accuracy' : accuracy_score(y_test_arr, y_pred),
        'Precision': precision_score(y_test_arr, y_pred, zero_division=0),
        'Recall'   : recall_score(y_test_arr, y_pred, zero_division=0),
        'F1-Score' : f1_score(y_test_arr, y_pred, zero_division=0),
        'AUC-ROC'  : roc_auc_score(y_test_arr, y_prob)
    }

results_df      = pd.DataFrame(test_results).T
best_name       = results_df['F1-Score'].idxmax()

print('Test Set Results:')
display(
    results_df.round(4)
    .style.highlight_max(color='#c6efce')
    .highlight_min(color='#ffc7ce')
    .format('{:.4f}')
)
print(f'\n🏆 Best Model (by F1): {best_name}')

In [ ]:
# ─── 9.2  Classification Reports ──────────────────────────────────────────────
for name in models:
    print(f'\n{"─"*50}\n  {name}\n{"─"*50}')
    print(classification_report(y_test_arr, preds[name]['y_pred'],
                                  target_names=['Phishing','Legitimate']))

In [ ]:
# ─── 9.3  Confusion Matrices ──────────────────────────────────────────────────
n_m   = len(models)
n_col = 3
n_row = (n_m + n_col - 1) // n_col

fig, axes = plt.subplots(n_row, n_col, figsize=(16, n_row*5))
fig.suptitle('Confusion Matrices', fontsize=15)
axes = axes.flatten()

for i, name in enumerate(models):
    cm = confusion_matrix(y_test_arr, preds[name]['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Phishing','Legitimate']).plot(
        ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(
        f'{name}\nF1={results_df.loc[name,"F1-Score"]:.3f}  AUC={results_df.loc[name,"AUC-ROC"]:.3f}',
        fontsize=10)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ─── 9.4  ROC & Precision-Recall Curves ──────────────────────────────────────
palette = sns.color_palette('tab10', n_m)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Performance Curves — All Models', fontsize=14)

for i, name in enumerate(models):
    fpr, tpr, _ = roc_curve(y_test_arr, preds[name]['y_prob'])
    auc = test_results[name]['AUC-ROC']
    axes[0].plot(fpr, tpr, lw=2, color=palette[i],
                  label=f'{name.split()[0]} (AUC={auc:.3f})')

    prec, rec, _ = precision_recall_curve(y_test_arr, preds[name]['y_prob'])
    axes[1].plot(rec, prec, lw=2, color=palette[i], label=name.split()[0])

axes[0].plot([0,1],[0,1],'k--',lw=1, label='Random (0.500)')
axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC Curves')
axes[0].legend(loc='lower right', fontsize=9)

axes[1].axhline(y_test_arr.mean(), color='k', ls='--', lw=1,
                 label=f'Baseline ({y_test_arr.mean():.2f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves')
axes[1].legend(loc='lower left', fontsize=9)

plt.tight_layout(); plt.show()

In [ ]:
# ─── 9.5  Model Comparison Dashboard ─────────────────────────────────────────
metrics  = ['Accuracy','Precision','Recall','F1-Score','AUC-ROC']
s_names  = [n.split()[0] for n in results_df.index]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Model Comparison', fontsize=14)

# Grouped bar
x   = np.arange(len(metrics))
w   = 0.15
pal = sns.color_palette('husl', n_m)
for i, (name, row) in enumerate(results_df.iterrows()):
    axes[0].bar(x + i*w, row[metrics].values, w,
                 label=name.split()[0], color=pal[i], edgecolor='black', lw=0.4)
axes[0].set_xticks(x + w*(n_m-1)/2)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0.5, 1.05)
axes[0].axhline(0.9, color='gray', ls='--', lw=0.8)
axes[0].set_title('All Metrics by Model')
axes[0].legend(fontsize=9)

# Heatmap
hm = results_df[metrics].copy()
hm.index = s_names
sns.heatmap(hm.astype(float), annot=True, fmt='.3f', cmap='YlGn',
             ax=axes[1], linewidths=0.5, vmin=0.7, vmax=1.0,
             cbar_kws={'label':'Score'})
axes[1].set_title('Performance Heatmap')

plt.tight_layout(); plt.show()

In [ ]:
# ─── 9.6  Feature Importance of Best Model ────────────────────────────────────
best_model = trained[best_name]

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=selected_features).sort_values()
    fig, ax = plt.subplots(figsize=(9, max(5, len(imp)*0.35)))
    clrs = [C['phish'] if i >= len(imp)-5 else C['blue'] for i in range(len(imp))]
    imp.plot(kind='barh', ax=ax, color=clrs, edgecolor='black', lw=0.3)
    ax.axvline(imp.mean(), color='red', ls='--', lw=1.5, label='Mean')
    ax.set_title(f'Feature Importance — {best_name}\n(on the {len(selected_features)} selected features)',
                  fontsize=12)
    ax.set_xlabel('Importance Score')
    ax.legend()
    plt.tight_layout(); plt.show()
elif hasattr(best_model, 'coef_'):
    coef = pd.Series(np.abs(best_model.coef_[0]), index=selected_features).sort_values()
    fig, ax = plt.subplots(figsize=(9, max(5, len(coef)*0.35)))
    coef.plot(kind='barh', ax=ax, color=C['blue'], edgecolor='black', lw=0.3)
    ax.set_title(f'|Coefficients| — {best_name}', fontsize=12)
    ax.set_xlabel('|Coefficient|')
    plt.tight_layout(); plt.show()

In [ ]:
# ─── 9.7  Learning Curves ─────────────────────────────────────────────────────
best_info = models[best_name]

train_sizes, tr_scores, val_scores = learning_curve(
    trained[best_name], best_info['Xtr'], y_train_sel,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='f1', n_jobs=-1
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Learning Curves — {best_name}', fontsize=13)

for ax, (tr, va, title) in zip(axes, [
    (tr_scores, val_scores, 'F1 Learning Curve'),
]):
    ax.plot(train_sizes, tr.mean(1), 'o-', color=C['legit'], lw=2.2, label='Train F1')
    ax.fill_between(train_sizes, tr.mean(1)-tr.std(1), tr.mean(1)+tr.std(1),
                     alpha=0.12, color=C['legit'])
    ax.plot(train_sizes, va.mean(1), 'o-', color=C['phish'], lw=2.2, label='Val F1')
    ax.fill_between(train_sizes, va.mean(1)-va.std(1), va.mean(1)+va.std(1),
                     alpha=0.12, color=C['phish'])
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('F1-Score')
    ax.set_title(title)
    ax.legend()
    ax.set_ylim(0.3, 1.05)

# CV fold distribution (10-fold boxplot)
detail = cross_validate(
    trained[best_name], best_info['Xtr'], y_train_sel,
    cv=StratifiedKFold(10, shuffle=True, random_state=RANDOM_STATE),
    scoring=['accuracy','f1','roc_auc']
)
bp = axes[1].boxplot(
    [detail['test_accuracy'], detail['test_f1'], detail['test_roc_auc']],
    patch_artist=True, labels=['Accuracy','F1','AUC-ROC'],
    medianprops=dict(color='black', lw=2)
)
for patch, col in zip(bp['boxes'], [C['blue'],C['phish'],C['legit']]):
    patch.set_facecolor(col+'80')
axes[1].set_title('10-Fold CV Score Distribution')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0.5, 1.05)

plt.tight_layout(); plt.show()

In [ ]:
# ─── 9.8  Overfitting Analysis ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Overfitting Analysis', fontsize=13)

short = [n.split()[0] for n in cv_df.index]
x_pos = np.arange(len(cv_df))

axes[0].bar(x_pos-0.2, cv_df['CV Accuracy'], 0.35, label='CV Acc', color=C['blue'], edgecolor='black')
axes[0].bar(x_pos+0.2, results_df['Accuracy'].values, 0.35, label='Test Acc', color=C['phish'], edgecolor='black')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(short, rotation=15)
axes[0].set_ylim(0.5, 1.05); axes[0].legend(); axes[0].set_title('CV vs Test Accuracy')

gaps   = cv_df['Overfit Gap'].values
clrs_g = [C['phish'] if g > 0.05 else C['legit'] for g in gaps]
axes[1].bar(short, gaps, color=clrs_g, edgecolor='black')
axes[1].axhline(0.05, color='red', ls='--', lw=1.5, label='Threshold (0.05)')
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Overfit Gap  (Train Acc − CV Acc)')
axes[1].set_ylabel('Gap'); axes[1].legend()

plt.tight_layout(); plt.show()

for name, g in zip(cv_df.index, gaps):
    status = '⚠️ OVERFIT' if g > 0.05 else '✅ OK'
    print(f'{name:<35}  Gap={g:.4f}  {status}')

---
## 10. 🏆 Final Summary

In [ ]:
# ─── 10.1  Print Summary ──────────────────────────────────────────────────────
best = results_df.loc[best_name]

print('╔══════════════════════════════════════════════════════╗')
print('║          FINAL PIPELINE SUMMARY                     ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Dataset rows         : {len(df_clean):,}                        ║'[:56]+'║')
print(f'║  Original features    : {n_features}                         ║'[:56]+'║')
print(f'║  Selected features    : {len(selected_features)}  (CoD-safe ratio: {len(X_train_sel)/len(selected_features):.0f}:1)  ║'[:56]+'║')
print(f'║  Train / Test split   : 80% / 20%                   ║')
print(f'║  Imbalance handling   : {"SMOTE" if USE_SMOTE else "class_weight=balanced"}                 ║'[:56]+'║')
print('╠══════════════════════════════════════════════════════╣')
print('║  MODEL RESULTS (Test Set)                           ║')
print('╠══════════════════════════════════════════════════════╣')
for name, row in results_df.iterrows():
    mark = '🏆' if name == best_name else '  '
    print(f'║ {mark} {name[:18]:<18} F1={row["F1-Score"]:.3f}  AUC={row["AUC-ROC"]:.3f} ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  🥇 Best : {best_name[:20]:<20}                 ║'[:56]+'║')
print(f'║  Accuracy  : {best["Accuracy"]:.4f}                               ║'[:56]+'║')
print(f'║  Precision : {best["Precision"]:.4f}                               ║'[:56]+'║')
print(f'║  Recall    : {best["Recall"]:.4f}                               ║'[:56]+'║')
print(f'║  F1-Score  : {best["F1-Score"]:.4f}                               ║'[:56]+'║')
print(f'║  AUC-ROC   : {best["AUC-ROC"]:.4f}                               ║'[:56]+'║')
print('╚══════════════════════════════════════════════════════╝')

In [ ]:
# ─── 10.2  Save Results ───────────────────────────────────────────────────────
results_df.to_csv('test_results.csv')
cv_df.to_csv('cv_results.csv')
pd.DataFrame({'selected_features': selected_features}).to_csv('selected_features.csv', index=False)

print('✅ test_results.csv')
print('✅ cv_results.csv')
print('✅ selected_features.csv')
print('\n🎉 Pipeline complete!')